# FID (Fréchet Inception Distance) Evaluation

This notebook evaluates diffusion model-generated CIFAR-10 images using the **Overall FID** metric computed on the full dataset (all classes combined).

## Important Notes
- **Overall FID is the standard metric** used in research papers
- FID uses InceptionV3 features (2048-dimensional) from the final pooling layer
- Lower FID = better quality and similarity to real data
- Uses the `clean-fid` library for robust and standardized FID calculation

## CIFAR-10 Classes
0: airplane, 1: automobile, 2: bird, 3: cat, 4: deer,
5: dog, 6: frog, 7: horse, 8: ship, 9: truck

## Imports and Setup

In [ ]:
# Standard library imports
import os

# Third-party imports
import torch
from torchvision import datasets
from tqdm import tqdm

## Device Configuration

In [ ]:
# Select device (MPS for Apple Silicon, CUDA for NVIDIA, CPU as fallback)
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

## Download CIFAR-10 Test Set

**⚠️ IMPORTANT:** If you have already run this cell and the `cifar10_real/` folder exists with 10,000 images, you can **skip this cell** and proceed directly to **"Configuration"**.

This cell downloads the CIFAR-10 test set and exports all 10,000 images to the `cifar10_real/` folder.

In [ ]:
real_folder = "cifar10_real"
os.makedirs(real_folder, exist_ok=True)

print("Loading CIFAR-10 test set...")
dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=None)

print(f"Start exporting {len(dataset)} CIFAR-10 test images to {real_folder}...")

for idx, (img_pil, label) in enumerate(tqdm(dataset, desc="Exporting images")):
    fname = f"{label}_{idx:05d}.png"  
    img_pil.save(os.path.join(real_folder, fname))

print("Export complete.")

## Configuration

In [ ]:
# Folders to evaluate
real_folder = "cifar10_real"

# All folders for overall FID (includes uncond)
folders_overall_fid = [
    "cond_input",
    "cond_time",
    "uncond",
    "cfg_eval_w1.0",
    "cfg_eval_w8",
    "cfg_eval_w3.0",
    "cfg_eval_w12.0",
    "cfg_eval_w5.0",
]

## Overall FID Calculation

Computes FID between real CIFAR-10 test set and each generated image folder.
- Uses `clean-fid` library for standardized FID calculation
- All classes combined for stable estimation
- Primary metric for evaluating generated image quality

In [ ]:
from cleanfid import fid

print("="*70)
print("OVERALL FID CALCULATION (FULL DATASET)")
print("="*70)
print("This is the standard FID calculation used in research papers.")
print("All classes are combined for more stable covariance estimation.\n")
print("Using clean-fid library for robust FID calculation.\n")

# Calculate FID for each generated folder using clean-fid
overall_fid_results = {}

for folder in folders_overall_fid:
    print(f"\n{'='*70}")
    print(f"Processing: {folder}")
    print(f"{'='*70}")
    
    gen_files = [f for f in os.listdir(folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"  Found {len(gen_files)} generated images")
    
    if len(gen_files) < 50:
        print(f"  ⚠️  WARNING: Only {len(gen_files)} images. FID is most reliable with 1000+ images.")
    
    try:
        # One-liner FID calculation using clean-fid

        fid_score = fid.compute_fid(
            fdir1=folder,
            fdir2=real_folder,
            # dataset_name="cifar10", 
            # dataset_res=32,
            # dataset_split="test",
            device=device,
            num_workers=0,
            use_dataparallel=False
        )
        overall_fid_results[folder] = fid_score
        print(f"  📊 Overall FID: {fid_score:.2f}")
        
    except Exception as e:
        print(f"  ❌ Failed to compute FID: {e}")
        overall_fid_results[folder] = None


# Summary
print("\n" + "="*70)
print("OVERALL FID SUMMARY (lower is better)")
print("="*70)

valid_results = {k: v for k, v in overall_fid_results.items() if v is not None}
sorted_results = sorted(valid_results.items(), key=lambda x: x[1])

for i, (folder, score) in enumerate(sorted_results, 1):
    medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{medal} {folder:20s}: FID = {score:7.2f}")

for folder, score in overall_fid_results.items():
    if score is None:
        print(f"❌ {folder:20s}: Calculation failed")